# Diffusion-MRI Image Synthesis with a Multi-GPU 3D U-Net

This notebook trains a 3D U-Net to synthesize one diffusion-MRI-derived map
from another (here: predicting a **Restricted Normalized Isotropic (RNI)**
diffusion map from a **Fractional Anisotropy (FA)** map). Image synthesis
lets a modality that is expensive, slow to acquire, or missing for some
participants be estimated directly from a routinely-acquired scan.

**What this notebook demonstrates:**
- A configurable 2D/3D U-Net built from a YAML spec (depth, filters, skip
  connections are not hard-coded)
- A leakage-safe, **grouped** cross-validation design (family-level
  grouping, so no participant's relatives appear in both train and test)
- A fully held-out **external test set**, evaluated only once, as a
  5-model ensemble
- **Multi-GPU training across independent SLURM jobs**: the 5
  cross-validation folds train in parallel, one per GPU, orchestrated by a
  three-phase job-dependency chain (`slurm/submit_all.sh`) with
  epoch-level checkpointing so a fold resumes cleanly if its job is
  preempted
- Quantitative evaluation (MAE, RMSE, PSNR, SSIM, correlation) plus a
  sensitivity analysis of how those metrics vary across folds

**Note on data:** the dataset used for this project is drawn from a
restricted-access neuroimaging study and is not distributed with this
repository (see the README for the expected data layout). Running this
notebook end-to-end requires supplying your own paired imaging data in
that format.


## 1. Setup

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import torch
import yaml

# Run notebook from the repo root so `src/` and `config/` resolve correctly,
# regardless of the working directory nbconvert launches it from.
REPO_ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

from src import build_unet, train_Unet

print(f"Repository root : {REPO_ROOT}")
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count        : {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


## 2. Configuration

All data paths, model architecture, and training hyperparameters live in
[`config/config.yaml`](../config/config.yaml) -- nothing below is hard-coded,
so a new experiment (different modalities, a 2D instead of 3D model, a
different loss) only requires editing the config.


In [ ]:
config_path = os.getenv('CONFIG_PATH', 'config/config.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

input_modality = config['input_modality']
output_modality = config['output_modality']
drop_missing_modalities = config['Drop_missing_modalities_flag']

data_directory = config['data_directory']
data_frame = config['data_frame']
samples_reserved_to_external_test = config['samples_reserved_to_external_test']

time_point_flag = config['time_point_flag']
time_point = set(config['time_point'])

manufacture_flag = config['manufacture_flag']
manufacture = set(config['manufacture'])
manufacture_filter = set(config['manufacture_filter'])

unet_config = config['model_info']['unet']
in_channels = unet_config['in_channels']
out_channels = unet_config['out_channels']
dimensions = unet_config['dimensions']
input_shape = unet_config['input_shape']

encoder_layers = config['model_info']['encoder_layers']
bottleneck = config['model_info']['bottleneck']
decoder_layers = config['model_info']['decoder_layers']
output_layer = config['model_info']['output_layer']

n_folds = config['n_folds']
fold_strategy = config['fold_strategy']
train_ratio = config['train_ratio']
val_ratio = config['val_ratio']
test_ratio = config['test_ratio']
batch_size = config['batch_size']
n_epochs = config['n_epochs']
learning_rate = config['learning_rate']
weight_decay = config['weight_decay']
optimizer_type = config['optimizer_type']
loss_function = config['loss_function']
loss_weights = config['loss_weights']
scheduler_type = config['scheduler_type']
scheduler_params = config['scheduler_params']
clip_grad = config['clip_grad']
early_stopping_patience = config['early_stopping_patience']
augmentation_config = config['augmentation_config']

slice_axis = config['slice_axis']
skip_background_slices = config['skip_background_slices']

print(f"Task            : {input_modality} -> {output_modality}")
print(f"Model            : {'3D' if dimensions == 3 else '2D'} U-Net, input shape {input_shape}")
print(f"Cross-validation : {n_folds}-fold, grouped by '{fold_strategy}'")
print(f"External test    : {samples_reserved_to_external_test} held-out samples")


## 3. Load Data

Each modality is stored as one stacked `.npy` volume array plus an index
CSV that aligns array rows with participant metadata. Participant-level
metadata (family ID, site, imaging session, etc.) drives the grouped
cross-validation split below.


In [ ]:
modalities = input_modality + output_modality
print(f"Loading modalities: {modalities}")

metadata = pd.read_csv(os.path.join(data_directory, data_frame))

image_volumes = [None] * len(modalities)
modality_index = [None] * len(modalities)

for i, modality in enumerate(modalities):
    image_volumes[i] = np.load(os.path.join(data_directory, f'{modality}.npy'))
    modality_index[i] = pd.read_csv(os.path.join(data_directory, f'{modality}_index.csv'))
    print(f"  {modality}: {image_volumes[i].shape}")

print(f"\nParticipant metadata: {metadata.shape[0]} rows, {metadata.shape[1]} columns")


## 4. Preprocess: Quality Filtering

Two filters are applied before any train/test split is made, so they
cannot leak information between splits:

1. **Missing-modality filtering** -- drop any participant where a modality
   volume is empty (zero mean) or its index row is degenerate.
2. **Cohort filtering** -- optionally restrict to specific imaging visits
   and/or scanner manufacturers, for a more homogeneous training cohort.


In [ ]:
print("Checking for missing or invalid modality volumes...")

if drop_missing_modalities:
    from multiprocessing import Pool

    def volume_mean(args):
        volume, idx = args
        return float(np.mean(volume[idx]))

    modality_masks = []
    for m, modality in enumerate(modalities):
        volume = image_volumes[m]
        with Pool(32) as pool:
            means = np.array(pool.map(volume_mean, [(volume, i) for i in range(len(volume))]))

        nonzero_mean = means != 0
        nonzero_index = (modality_index[m].values != 0).all(axis=1)
        mask = nonzero_mean & nonzero_index
        modality_masks.append(mask)
        print(f"  {modality}: {mask.sum()}/{len(mask)} valid samples")

    valid_mask = modality_masks[0]
    for mask in modality_masks[1:]:
        valid_mask &= mask
    valid_indices = np.where(valid_mask)[0]
    print(f"\nValid in ALL modalities: {len(valid_indices)}/{len(valid_mask)}")
else:
    valid_indices = np.arange(len(image_volumes[0]))


In [ ]:
print("Applying cohort filters (imaging visit / scanner manufacturer)...")

if manufacture_flag:
    mask = metadata[list(manufacture)].isin(manufacture_filter).all(axis=1)
    manufacturer_mask = np.where(mask)[0]
else:
    manufacturer_mask = np.arange(len(metadata))

if time_point_flag:
    mask = metadata['Imaging_session_id'].isin(time_point)
    visit_mask = np.where(mask)[0]
else:
    visit_mask = np.arange(len(metadata))

from functools import reduce
keep_indices = reduce(np.intersect1d, [manufacturer_mask, visit_mask, valid_indices])
keep_mask = np.isin(np.arange(len(image_volumes[0])), keep_indices)

image_volumes = [volume[keep_mask] for volume in image_volumes]
metadata = metadata.iloc[keep_mask].reset_index(drop=True)

print(f"\nSamples after filtering: {keep_mask.sum()} / {len(keep_mask)}")
for modality, volume in zip(modalities, image_volumes):
    print(f"  {modality}: {volume.shape}")


## 5. Grouping Columns for Cross-Validation

Splitting is done at the **family** level (or **site** level), never at
the individual-scan level: siblings/relatives share genetics and often
scanner/site, so splitting by scan alone would leak information between
train and test. `iid`/`eid`/`fid` identify the participant, imaging
session, and family; `sid` identifies the acquisition site.


In [ ]:
iid = metadata['Imaging_participant_id']
eid = metadata['Imaging_session_id']
fid = metadata['rel_family_id']
sid = metadata['ab_g_dyn__design_site']

print(f"Unique participants : {iid.nunique()}")
print(f"Unique families      : {fid.nunique()}")
print(f"Unique sites          : {sid.nunique()}")


## 6. Build the Model

In [ ]:
print("Building U-Net from config...")

model = build_unet(
    in_channels=in_channels,
    out_channels=out_channels,
    dimensions=dimensions,
    input_shape=input_shape,
    encoder_layers=encoder_layers,
    bottleneck=bottleneck,
    decoder_layers=decoder_layers,
    output_layer=output_layer,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"\nModel moved to device: {device}")


## 7. Training Configuration

`experiment_name` must be identical across all three SLURM phases
(preprocess / train / aggregate) -- it is how they find each other's
output on disk.


In [ ]:
input_str = '-'.join(input_modality)
output_str = '-'.join(output_modality)
experiment_name = f"Input-{input_str}__Output-{output_str}"

# Grouping column must match fold_strategy, otherwise cross-validation
# would silently group by the wrong variable.
if fold_strategy == 'site':
    fold_groups = pd.DataFrame({'ab_g_dyn__design_site': sid})
else:
    fold_groups = pd.DataFrame({'rel_family_id': fid})

print(f"Experiment name : {experiment_name}")
print(f"Fold strategy   : {fold_strategy} (grouping column: {fold_groups.columns[0]})")


## 8. Multi-GPU Training

Training is split into three phases so it can run as three separate SLURM
jobs instead of one long single-GPU job:

| Phase | What it does | Resources |
|---|---|---|
| 1. `preprocess` | Split data, save fold indices to disk | CPU only, runs once |
| 2. `train_fold` | Train **one** cross-validation fold | 1 GPU, 5 jobs run in parallel |
| 3. `aggregate`  | Load all 5 fold models, evaluate external test as an ensemble | 1 GPU, runs once |

`slurm/submit_all.sh` submits all three phases with SLURM `--dependency`
chains so Phase 2's job array only starts after Phase 1 succeeds, and
Phase 3 only starts after **all 5** Phase-2 tasks succeed. Each fold's
job checkpoints after every epoch, so if a job is killed by the cluster's
time limit it resumes from its last completed epoch instead of restarting.

The cell below reads the `TRAINING_PHASE` / `FOLD_ID` environment
variables that the SLURM scripts export, and calls `train_Unet` with the
arguments for that phase. Running the notebook manually with no
environment variables set (`TRAINING_PHASE` unset) falls back to training
all folds sequentially in a single process -- useful for local debugging
on a machine with one GPU.


In [ ]:
training_phase = os.environ.get('TRAINING_PHASE', 'all')
fold_id_str = os.environ.get('FOLD_ID', '')
fold_id = int(fold_id_str) if fold_id_str.strip() else None

print(f"TRAINING_PHASE : {training_phase}")
print(f"FOLD_ID        : {fold_id}")

shared_kwargs = dict(
    model=model,
    input_modality=image_volumes[0],
    output_modality=image_volumes[1],
    samples_reserved_to_external_test=samples_reserved_to_external_test,
    groups=fold_groups,
    iid=iid, eid=eid, fid=fid,
    n_folds=n_folds,
    fold_strategy=fold_strategy,
    train_ratio=train_ratio, val_ratio=val_ratio, test_ratio=test_ratio,
    batch_size=batch_size,
    n_epochs=n_epochs,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    optimizer_type=optimizer_type,
    loss_function=loss_function,
    loss_weights=loss_weights,
    scheduler_type=scheduler_type,
    scheduler_params=scheduler_params,
    clip_grad=clip_grad,
    early_stopping_patience=early_stopping_patience,
    augmentation_config=augmentation_config,
    device='cuda',
    save_dir='./experiments',
    experiment_name=experiment_name,
    random_state=42,
    verbose=True,
    save_all_fold_models=True,
    save_predictions=True,
    slice_axis=slice_axis,
    skip_background_slices=skip_background_slices,
)

if training_phase == 'preprocess':
    print("\n-- Phase 1: splitting data and saving fold indices --")
    results = train_Unet(**shared_kwargs, preprocess_only=True)

elif training_phase == 'train_fold':
    print(f"\n-- Phase 2: training fold {fold_id} --")
    results = train_Unet(**shared_kwargs, fold_id=fold_id)

elif training_phase == 'aggregate':
    print("\n-- Phase 3: loading models and evaluating external test set --")
    results = train_Unet(**shared_kwargs, aggregate_only=True)

else:
    print("\n-- Single-job mode: training all folds sequentially --")
    results = train_Unet(**shared_kwargs)


## 9. Training Results

Each fold saves its own loss curves, validation metrics, and sample
predictions to `experiments/<experiment_name>/fold_<n>/`. Once all folds
have run (either sequentially above, or as separate SLURM jobs), the
per-fold summary is available in `fold_results.csv`.


In [ ]:
fold_results_path = os.path.join('experiments', experiment_name, 'fold_results.csv')

if os.path.exists(fold_results_path):
    fold_results = pd.read_csv(fold_results_path)
    print(fold_results)
else:
    print(f"No fold_results.csv yet at {fold_results_path}")
    print("(expected once all folds have completed training)")


## 10. Evaluation: Accuracy and Sensitivity Analysis

The aggregation phase evaluates the **external test set** -- samples that
were held out before cross-validation ever started, and never used for
model selection or early stopping -- with each of the 5 fold models, and
with their ensemble average.

Two questions this answers:

- **Accuracy**: how close is the ensemble's prediction to the true target
  map, measured by MAE, RMSE, PSNR, SSIM, and voxel-wise correlation?
- **Sensitivity (robustness)**: how much does performance vary across the
  5 independently-trained folds? A model whose fold-to-fold metrics are
  tightly clustered is more trustworthy than one with high variance, even
  if the average looks similar -- high variance suggests sensitivity to
  which participants happened to be in the training set.


In [ ]:
external_dir = os.path.join('experiments', experiment_name, 'external_test')

if os.path.exists(external_dir):
    ensemble_preds = np.load(os.path.join(external_dir, 'predictions_ensemble.npy'))
    targets = np.load(os.path.join(external_dir, 'target.npy'))

    from src.metrics import compute_metrics
    ensemble_metrics = compute_metrics(ensemble_preds, targets)

    print("Ensemble performance on the external test set:")
    for metric, value in ensemble_metrics.items():
        print(f"  {metric.upper()}: {value:.4f}")
else:
    print(f"No external test predictions yet at {external_dir}")
    print("(produced by the aggregation phase, after all folds have trained)")


In [ ]:
if 'fold_results' in dir() and not fold_results.empty:
    print("Per-fold test-set metrics (mean +/- std across folds):")
    print("A small std relative to the mean indicates the model's accuracy")
    print("is not overly sensitive to which participants were used to train it.\n")

    import ast
    test_metrics = pd.json_normalize(fold_results['test_metrics'].apply(ast.literal_eval))
    summary = test_metrics.agg(['mean', 'std']).T
    print(summary)


## 11. Interpretation and Conclusions

- **Data leakage controls**: the external test set and all cross-validation
  splits are made at the family level, with an explicit overlap check
  (`_check_overlap`) run before any training starts. The external test set
  is only ever scored once, by the aggregation phase, after every fold's
  hyperparameters and early-stopping decisions have already been made.
- **Ensembling**: the final prediction averages 5 independently-trained
  models (one per cross-validation fold), which reduces variance compared
  to any single fold's model.
- **Multi-GPU scaling**: because each fold is an independent SLURM job,
  wall-clock training time for the full 5-fold cross-validation is close
  to the time for *one* fold, not five, as long as 5 GPUs are available
  concurrently -- the job-array design in `slurm/02_train_fold.sh` is what
  makes that possible.
- **What to look for in the metrics above**: MAE/RMSE/PSNR summarize
  voxel-wise reconstruction accuracy; SSIM and correlation summarize how
  well spatial structure is preserved, which matters more than raw
  intensity error for a map that will be interpreted visually.

This notebook is the single source of truth for the experiment: the same
code path runs whether it's executed manually end-to-end, or split across
the three SLURM phases in `slurm/`.
